In [ ]:
!pip install pandas numpy scikit-learn spacy transformers rapidfuzz python-pptx langdetect nltk
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 58.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import StandardScaler
from langdetect import detect
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize, word_tokenize
import spacy
from rapidfuzz import process
from transformers import pipeline
from pptx import Presentation
from datetime import datetime

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
#DATA INGESTION

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Infosys_D/customer_support_tickets.csv")
print(df.shape)
df.head()

(8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [ ]:
#1️⃣ DATA CLEANING

In [ ]:
#Missing Values

df.isnull().sum()
df.fillna("UNKNOWN", inplace=True)

/tmp/ipython-input-1979965605.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'UNKNOWN' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("UNKNOWN", inplace=True)


In [ ]:
df.fillna("UNKNOWN", inplace=True)

In [ ]:
# convert all columns to string type before fillna
df = df.astype("string")      # uses pandas dedicated string dtype
df.fillna("UNKNOWN", inplace=True)

In [ ]:
# only fill string columns with a string
for col in df.select_dtypes(include="object"):
    df[col].fillna("UNKNOWN", inplace=True)

# fill numeric columns with e.g. 0 or another numeric sentinel
for col in df.select_dtypes(include=["float64","int64"]):
    df[col].fillna(0, inplace=True)

In [ ]:
df["Customer Name"] = df["Customer Name"].astype("string").fillna("UNKNOWN")

In [ ]:
#Remove Duplicates

before = len(df)
df.drop_duplicates(inplace=True)
print("Removed:", before - len(df))

Removed: 0


In [ ]:
#Fix Formatting

df.columns = df.columns.str.lower().str.replace(" ", "_")
df.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,UNKNOWN,Critical,Social media,2023-06-01 12:15:36,UNKNOWN,UNKNOWN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,UNKNOWN,Critical,Chat,2023-06-01 16:45:38,UNKNOWN,UNKNOWN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [ ]:
#2️⃣ DATA VALIDATION

In [ ]:
assert "ticket_id" in df.columns
assert df["ticket_id"].notnull().all()

In [ ]:
print(df.dtypes)

ticket_id                       string[python]
customer_name                   string[python]
customer_email                  string[python]
customer_age                    string[python]
customer_gender                 string[python]
product_purchased               string[python]
date_of_purchase                string[python]
ticket_type                     string[python]
ticket_subject                  string[python]
ticket_description              string[python]
ticket_status                   string[python]
resolution                      string[python]
ticket_priority                 string[python]
ticket_channel                  string[python]
first_response_time             string[python]
time_to_resolution              string[python]
customer_satisfaction_rating    string[python]
dtype: object


In [ ]:
#3️⃣ DATA TRANSFORMATION

In [ ]:
if "created_at" in df.columns:
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")

In [ ]:
scaler = StandardScaler()
if "priority" in df.columns:
    df["priority_scaled"] = scaler.fit_transform(df[["priority"]])

In [ ]:
#4️⃣ DATA FILTERING

In [ ]:
df = df[df["ticket_description"] != "UNKNOWN"].copy()
df.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,UNKNOWN,Critical,Social media,2023-06-01 12:15:36,UNKNOWN,UNKNOWN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,UNKNOWN,Critical,Chat,2023-06-01 16:45:38,UNKNOWN,UNKNOWN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [ ]:
#5️⃣ DATA ENRICHMENT

In [ ]:
df["text_length"] = df["ticket_description"].apply(len)
df["word_count"] = df["ticket_description"].apply(lambda x: len(x.split()))
df.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating,text_length,word_count
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,UNKNOWN,Critical,Social media,2023-06-01 12:15:36,UNKNOWN,UNKNOWN,284,43
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,UNKNOWN,Critical,Chat,2023-06-01 16:45:38,UNKNOWN,UNKNOWN,282,44
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,275,42
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,262,41
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,333,55


In [ ]:
#6️⃣ DATA DEDUPLICATION

In [ ]:
df = df.drop_duplicates(subset=["ticket_description"])

In [ ]:
#7️⃣ DATA MASKING

In [ ]:
def mask_email(text):
    return re.sub(r'\S+@\S+', '[EMAIL]', text)

df["ticket_description"] = df["ticket_description"].apply(mask_email)

In [ ]:
# df = df[df["ticket_status"] == "open"] # Commenting out this filter, as it was emptying the DataFrame

In [ ]:
df["ticket_description"] = df["ticket_description"].apply(mask_email)

In [ ]:
df.loc[:, "ticket_description"] = df["ticket_description"].apply(mask_email)

In [ ]:
df["ticket_description"] = df["ticket_description"].apply(mask_email)

In [ ]:
df = df.copy()
df.loc[:, "ticket_description"] = df["ticket_description"].apply(mask_email)

In [ ]:
def mask_email(text):
    return re.sub(r'\S+@\S+', '[EMAIL]', text)

In [ ]:
def mask_email(text):
    return re.sub(r'\S+@\S+', '[EMAIL]', text)

df["ticket_description"] = df["ticket_description"].apply(mask_email)

In [ ]:
#8️⃣ DATA STANDARDIZATION

In [ ]:
df["ticket_status"] = df["ticket_status"].str.upper()

In [ ]:
#9️⃣ ERROR HANDLING & LOGGING

In [ ]:
errors = []
for i,row in df.iterrows():
    try:
        len(row["description"])
    except:
        errors.append(i)
print("Error rows:", errors)

Error rows: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 21

In [ ]:
#🔟 METADATA HANDLING

In [ ]:
df["ingested_at"] = datetime.now()
df["source"] = "Kaggle Customer Support Dataset"

In [ ]:
#1️⃣1️⃣ SAMPLING

In [ ]:
sample_df = df.sample(frac=0.2, random_state=42)
sample_df.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,...,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating,text_length,word_count,ingested_at,source
1653,1654,Alison Larsen,kneal@example.net,57,Female,Google Pixel,2021-06-16,Billing inquiry,Delivery problem,The {product_purchased} is unable to establish...,...,Gas your culture include second.,Low,Social media,2023-06-01 10:17:32,2023-05-31 22:42:32,4.0,351,54,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset
7400,7401,Carrie Williams,joshua47@example.com,27,Male,Autodesk AutoCAD,2020-01-30,Billing inquiry,Network problem,I'm having an issue with the {product_purchase...,...,UNKNOWN,Low,Email,UNKNOWN,UNKNOWN,UNKNOWN,358,55,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset
2198,2199,John Hale,moralesjeffrey@example.net,49,Male,iPhone,2020-08-11,Billing inquiry,Account access,I've accidentally deleted important data from ...,...,UNKNOWN,Critical,Social media,UNKNOWN,UNKNOWN,UNKNOWN,363,60,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset
6529,6530,Leslie Duffy,amydunn@example.net,27,Other,LG OLED,2021-11-05,Technical issue,Display issue,I'm having an issue with the {product_purchase...,...,Marriage season green also cell.,Medium,Phone,2023-06-01 19:24:34,2023-06-01 17:20:34,1.0,290,48,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset
1937,1938,Jennifer Mclean,holmesthomas@example.org,48,Male,Dell XPS,2021-09-04,Refund request,Product recommendation,I'm having an issue with the {product_purchase...,...,UNKNOWN,High,Social media,UNKNOWN,UNKNOWN,UNKNOWN,296,48,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset


In [ ]:
#STAGE 1 – TEXT INGESTION

In [ ]:
df["language"] = df["ticket_description"].apply(lambda x: detect(x) if x!="UNKNOWN" else "unknown")
df["sentences"] = df["ticket_description"].apply(sent_tokenize)

In [ ]:
df.head()

,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,...,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating,text_length,word_count,ingested_at,source,language,sentences
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,...,Social media,2023-06-01 12:15:36,UNKNOWN,UNKNOWN,284,43,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset,en,[I'm having an issue with the {product_purchas...
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,...,Chat,2023-06-01 16:45:38,UNKNOWN,UNKNOWN,282,44,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset,en,[I'm having an issue with the {product_purchas...
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,...,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0,275,42,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset,en,[I'm facing a problem with my {product_purchas...
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,...,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0,262,41,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset,en,[I'm having an issue with the {product_purchas...
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,...,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0,333,55,2026-01-21 06:59:58.440551,Kaggle Customer Support Dataset,en,[I'm having an issue with the {product_purchas...


In [ ]:
df.describe()

,text_length,word_count,ingested_at
count,8077.000000,8077.000000,8077
mean,295.305311,47.436796,2026-01-21 06:59:58.440551
min,151.000000,21.000000,2026-01-21 06:59:58.440551
25%,278.000000,44.000000,2026-01-21 06:59:58.440551
50%,300.000000,49.000000,2026-01-21 06:59:58.440551
75%,319.000000,53.000000,2026-01-21 06:59:58.440551
max,397.000000,63.000000,2026-01-21 06:59:58.440551
std,36.086245,7.320994,NaN


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8077 entries, 0 to 8468
Data columns (total 23 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   ticket_id                     8077 non-null   string        
 1   customer_name                 8077 non-null   string        
 2   customer_email                8077 non-null   string        
 3   customer_age                  8077 non-null   string        
 4   customer_gender               8077 non-null   string        
 5   product_purchased             8077 non-null   string        
 6   date_of_purchase              8077 non-null   string        
 7   ticket_type                   8077 non-null   string        
 8   ticket_subject                8077 non-null   string        
 9   ticket_description            8077 non-null   object        
 10  ticket_status                 8077 non-null   string        
 11  resolution                    8077 

In [ ]:
#STAGE 2 – LLM-BASED NER

In [ ]:
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

df["entities"] = df["ticket_description"].apply(extract_entities)
df[["ticket_description","entities"]].head()

,ticket_description,entities
0,I'm having an issue with the {product_purchase...,"[(71701, CARDINAL)]"
1,I'm having an issue with the {product_purchase...,[]
2,I'm facing a problem with my {product_purchase...,"[(yesterday, DATE)]"
3,I'm having an issue with the {product_purchase...,[]
4,I'm having an issue with the {product_purchase...,[]


In [ ]:
#STAGE 3 – ENTITY NORMALIZATION

In [ ]:
canonical = {}

def normalize_entity(ent):
    if ent in canonical:
        return canonical[ent]
    best = process.extractOne(ent, canonical.keys())
    if best and best[1] > 85:
        canonical[ent] = canonical[best[0]]
    else:
        canonical[ent] = ent.lower()
    return canonical[ent]

df["normalized_entities"] = df["entities"].apply(
    lambda ents: [(normalize_entity(e),t) for e,t in ents]
)

In [ ]:
#STAGE 4 – RELATION EXTRACTION (LLM Prompting)

In [ ]:
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

relations = []
for text in df["ticket_description"].head(20):
    if len(text.split())>5:
        out = classifier(text, candidate_labels=["works_for","uses","located_in","depends_on","causes"])
        relations.append(out)
relations[:2]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


[{'sequence': "I'm having an issue with the {product_purchased}. Please assist.\n\nYour billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.",
  'labels': ['located_in', 'uses', 'causes', 'depends_on', 'works_for'],
  'scores': [0.29118379950523376,
   0.2495344877243042,
   0.2381872832775116,
   0.15540046989917755,
   0.06569398939609528]},
 {'sequence': "I'm having an issue with the {product_purchased}. Please assist.\n\nIf you need to change an existing product.\n\nI'm having an issue with the {product_purchased}. Please assist.\n\nIf The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",
  'labels': ['depends_on', 'located_in', 'uses', 'causes', 'works_for'],
  'scores': [0.34334781765937805,
   0.23036356270313263,
   0.21715635061264038,
   0.13254781067371368,
   0

In [ ]:
#TRIPLE CONSTRUCTION

In [ ]:
triples = []
for i,row in df.iterrows():
    for ent,typ in row["normalized_entities"]:
        triples.append((row["ticket_id"], ent, typ))
triples[:5]

[('1', '71701', 'CARDINAL'),
 ('3', 'yesterday', 'DATE'),
 ('6', 'yesterday', 'DATE'),
 ('7', "invalid credentials'", 'WORK_OF_ART'),
 ('11', '1-800', 'CARDINAL')]

In [ ]:
#POST PROCESSING

In [ ]:
final_triples = list(set(triples))
len(final_triples)

5484

In [ ]:
df = pd.read_csv("/content/drive/MyDrive/Infosys_D/customer_support_tickets.csv")
print(df.shape)
df.head()

(8469, 17)


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [ ]:
df.isnull().sum()

,0
Ticket ID,0
Customer Name,0
Customer Email,0
Customer Age,0
Customer Gender,0
Product Purchased,0
Date of Purchase,0
Ticket Type,0
Ticket Subject,0
Ticket Description,0


In [ ]:
df.fillna("UNKNOWN", inplace=True)

/tmp/ipython-input-2372803106.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'UNKNOWN' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.fillna("UNKNOWN", inplace=True)


In [ ]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='object')

In [ ]:
df.columns = df.columns.str.lower().str.replace(" ", "_")
df = df[df["ticket_description"] != "UNKNOWN"]

In [ ]:
print(df.shape)

(8469, 17)


In [ ]:
print("Rows:", len(df))
df.sample(5)

Rows: 8469


,ticket_id,customer_name,customer_email,customer_age,customer_gender,product_purchased,date_of_purchase,ticket_type,ticket_subject,ticket_description,ticket_status,resolution,ticket_priority,ticket_channel,first_response_time,time_to_resolution,customer_satisfaction_rating
7619,7620,Anne Deleon,michaelfrazier@example.com,52,Other,Bose SoundLink Speaker,2020-04-03,Product inquiry,Product compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,UNKNOWN,Medium,Phone,2023-06-01 11:33:40,UNKNOWN,UNKNOWN
5833,5834,Jake Murray,robert86@example.com,45,Female,Google Pixel,2021-12-26,Product inquiry,Cancellation request,I'm having an issue with the {product_purchase...,Closed,Protect federal seven challenge.,Low,Phone,2023-06-01 12:29:36,2023-06-01 21:27:36,5.0
3373,3374,Sherry Olsen,dawn90@example.com,54,Other,Amazon Echo,2020-08-13,Product inquiry,Product compatibility,"My {product_purchased} crashed, and I lost all...",Pending Customer Response,UNKNOWN,High,Chat,2023-06-01 22:13:28,UNKNOWN,UNKNOWN
6738,6739,Christopher Ellis,christinerichards@example.net,69,Male,Lenovo ThinkPad,2021-05-16,Technical issue,Software bug,I'm having an issue with the {product_purchase...,Closed,Water threat ground decade sound.,Medium,Email,2023-06-01 15:11:24,2023-06-01 00:38:24,5.0
2481,2482,Michael Kelly,morrisjeff@example.com,56,Other,Samsung Soundbar,2021-03-07,Product inquiry,Data loss,"I've recently set up my {product_purchased}, b...",Pending Customer Response,UNKNOWN,Critical,Phone,2023-06-01 05:54:00,UNKNOWN,UNKNOWN


In [ ]:
print(df.columns)
print(df.info())
print(df.head(3))

Index(['ticket_id', 'customer_name', 'customer_email', 'customer_age',
       'customer_gender', 'product_purchased', 'date_of_purchase',
       'ticket_type', 'ticket_subject', 'ticket_description', 'ticket_status',
       'resolution', 'ticket_priority', 'ticket_channel',
       'first_response_time', 'time_to_resolution',
       'customer_satisfaction_rating'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 17 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   ticket_id                     8469 non-null   int64 
 1   customer_name                 8469 non-null   object
 2   customer_email                8469 non-null   object
 3   customer_age                  8469 non-null   int64 
 4   customer_gender               8469 non-null   object
 5   product_purchased             8469 non-null   object
 6   date_of_purchase              8469 

In [ ]:
import networkx as nx

G = nx.DiGraph()

for _, r in df.iterrows():
    ticket = f"Ticket:{r['ticket_id']}"
    customer = f"Customer:{r['customer_name']}" # Changed from customer_id to customer_name
    # No direct 'agent_id' column, skipping for now or default to 'UNKNOWN_AGENT'
    agent = f"Agent:UNKNOWN_AGENT" # Placeholder as 'agent_id' column is not in df
    issue = f"Issue:{r['ticket_subject']}" # Changed from issue_type to ticket_subject
    product = f"Product:{r['product_purchased']}" # Changed from product to product_purchased

    G.add_node(ticket, type="Ticket")
    G.add_node(customer, type="Customer")
    G.add_node(agent, type="Agent")
    G.add_node(issue, type="Issue")
    G.add_node(product, type="Product")

    G.add_edge(customer, ticket, relation="CREATED")
    G.add_edge(agent, ticket, relation="HANDLED")
    G.add_edge(ticket, issue, relation="ABOUT")
    G.add_edge(ticket, product, relation="FOR_PRODUCT")

In [ ]:
nx.write_gml(G, "customer_support_kg.gml")

In [ ]:
edges = []
for u, v, d in G.edges(data=True):
    edges.append([u, d["relation"], v])

import pandas as pd
pd.DataFrame(edges, columns=["subject","predicate","object"]).to_csv("triples.csv", index=False)

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Infosys_D/customer_support_tickets.csv")
print("Shape:", df.shape)
print(df.head())

Shape: (8469, 17)
   Ticket ID        Customer Name              Customer Email  Customer Age  \
0          1        Marisa Obrien  carrollallison@example.com            32   
1          2         Jessica Rios    clarkeashley@example.com            42   
2          3  Christopher Robbins   gonzalestracy@example.com            48   
3          4     Christina Dillon    bradleyolson@example.org            27   
4          5    Alexander Carroll     bradleymark@example.com            67   

  Customer Gender Product Purchased Date of Purchase      Ticket Type  \
0           Other        GoPro Hero       2021-03-22  Technical issue   
1          Female       LG Smart TV       2021-05-22  Technical issue   
2           Other          Dell XPS       2020-07-14  Technical issue   
3          Female  Microsoft Office       2020-11-13  Billing inquiry   
4          Female  Autodesk AutoCAD       2020-02-04  Billing inquiry   

             Ticket Subject  \
0             Product setup   
1  Per

In [ ]:
display(df.head())

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [ ]:
import pandas as pd
import networkx as nx

df = pd.read_csv("/content/drive/MyDrive/Infosys_D/customer_support_tickets.csv")

# Standardize column names
df.columns = df.columns.str.lower().str.replace(" ", "_")

G = nx.DiGraph()

for _, r in df.iterrows():
    ticket = f"Ticket:{r['ticket_id']}"
    customer = f"Customer:{r['customer_name']}" # Use 'customer_name' instead of 'customer_id'
    agent = f"Agent:UNKNOWN_AGENT" # No 'agent_id' column, using a placeholder
    issue = f"Issue:{r['ticket_subject']}" # Use 'ticket_subject' instead of 'issue_type'
    product = f"Product:{r['product_purchased']}" # Use 'product_purchased' instead of 'product'

    G.add_node(ticket, type="Ticket")
    G.add_node(customer, type="Customer")
    G.add_node(agent, type="Agent")
    G.add_node(issue, type="Issue")
    G.add_node(product, type="Product")

    G.add_edge(customer, ticket, relation="CREATED")
    G.add_edge(agent, ticket, relation="HANDLED")
    G.add_edge(ticket, issue, relation="ABOUT")
    G.add_edge(ticket, product, relation="FOR_PRODUCT")

In [ ]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 16556
Edges: 33876


In [ ]:
for u, v, d in list(G.edges(data=True))[:10]:
    print(u, "--", d["relation"], "-->", v)

Ticket:1 -- ABOUT --> Issue:Product setup
Ticket:1 -- FOR_PRODUCT --> Product:GoPro Hero
Customer:Marisa Obrien -- CREATED --> Ticket:1
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:1
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:2
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:3
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:4
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:5
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:6
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:7


In [ ]:
nx.write_gml(G, "customer_support_kg.gml")
print("Knowledge graph saved as customer_support_kg.gml")

Knowledge graph saved as customer_support_kg.gml


In [ ]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Count node types
from collections import Counter
types = Counter([d.get("type") for _, d in G.nodes(data=True)])
print(types)

NameError: name 'G' is not defined

In [ ]:
import pandas as pd
import networkx as nx

# Load your CSV again
df = pd.read_csv("/content/drive/MyDrive/Infosys_D/customer_support_tickets.csv")

# Standardize column names
df.columns = df.columns.str.lower().str.replace(" ", "_")

# Build Knowledge Graph
G = nx.DiGraph()

for _, r in df.iterrows():
    ticket = f"Ticket:{r['ticket_id']}"
    customer = f"Customer:{r['customer_name']}" # Using 'customer_name' as 'customer_id' doesn't exist
    agent = f"Agent:UNKNOWN_AGENT" # Placeholder as 'agent_id' column is not in df
    issue = f"Issue:{r['ticket_subject']}" # Using 'ticket_subject' as 'issue_type' doesn't exist
    product = f"Product:{r['product_purchased']}" # Using 'product_purchased' as 'product' doesn't exist

    G.add_node(ticket, type="Ticket")
    G.add_node(customer, type="Customer")
    G.add_node(agent, type="Agent")
    G.add_node(issue, type="Issue")
    G.add_node(product, type="Product")

    G.add_edge(customer, ticket, relation="CREATED")
    G.add_edge(agent, ticket, relation="HANDLED")
    G.add_edge(ticket, issue, relation="ABOUT")
    G.add_edge(ticket, product, relation="FOR_PRODUCT")

In [ ]:
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 16556
Edges: 33876


In [ ]:
for u, v, d in list(G.edges(data=True))[:5]:
    print(u, "--", d["relation"], "-->", v)

Ticket:1 -- ABOUT --> Issue:Product setup
Ticket:1 -- FOR_PRODUCT --> Product:GoPro Hero
Customer:Marisa Obrien -- CREATED --> Ticket:1
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:1
Agent:UNKNOWN_AGENT -- HANDLED --> Ticket:2
